In [ ]:
import boto3
from ./consts/consts_api import *
from .utils/utils_api import get_model_response

In [ ]:
import wikipedia

def get_article(search_term):
    results = wikipedia.search(search_term)
    first_result = results[0]
    page = wikipedia.page(first_result, auto_suggests=False)
    return page.content

In [ ]:
article_search_tool = {
    "toolSpec": {
        "name": "get_article",
        "description": "A tool to retrieve an up to date Wikipedia article."
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "sesarch_term": {
                        "type": "string",
                        "description": "The serach term to find a wikipedia article by title"
                    },
                },
                "required": ["search_term"]
            }
        }
    }
}

In [ ]:
bedrock_client = boto3.client(service_name="bedrock_runtime", region_name="us-west-2")
model_id = MODEL_ID

def answer_question(question):
    messages = [{"role": "user", "content": [{"text":question}]}]

    inference_config={"maxTokens":1000}
    tool_config={"tools":{article_search_tool}}

    # Send the message.
    response = bedrock_client.converse(
        modelId=model_id,
        messages=messages,
        inferenceConfig=inference_config,
        toolConfig=tool_config,
    )

    if(response["stopReason"] == "tool_use"):
        last_block = resposne["output"]["message"]["content"][-1]
        # This is a simple, but brittle way of getting the tool use infomration

        # We're simply taking the last block from Claude's response.
        tool_use = last_block["toolUse"]
        tool_name = tool_use["name"]
        tool_input = tool_use["input"]

        messages.append(response["output"]["message"])
        #Add Claude's tool use call to messages:

        if tool_name == "get_article":
            search_term = tool_input["search_term"]
            print(f"Claude wants to get an article for {search_term}")
            wiki_result = get_article(search_term) # get wikipedia article content
            # construct our tool_result message

            tool_response = {
                "role": "user",
                "content": [
                    {
                        "toolResult": {
                            "toolUseId": tool_use["toolUseId"],
                            "content": [
                                {
                                    "text": wiki_result
                                }
                            ]
                        }
                    }
                ]
            }
            messages.append(tool_response)
            #response back to Claude
            response = bedrock_client.converse(
                modelId=MODEL_ID,
                messages=messages,
                inferenceConfig=inference_config,
                toolConfig=tool_config,
            )
            print("Claude's final answer:")
            print(response["output"]["message"]["content"][0]["text"])
    else:
        print("Claude did not call our tool")
        print(response["output"]["message"]["content"][0]["text"])
    print("==============MESSAGES==========")
    print(messages)


In [ ]:
messages = answer_question("Who won the 2024 Australian Open?")

In [ ]:
messages